# Dataset pipeline

Building a financial abductive reasoning benchmark: a market event, the news
around it, and four candidate causes.

| Stage | What it does | Cost |
|---|---|---|
| 1 | Find big move days from price data | free |
| 2 | Retrieve news + distractor documents | ~7 serper credits/event |
| 3 | Extract candidate causes | 1 gemini request/topic |
| 4 | Score candidates with 2+ models | 1 request per model per topic |
| 5 | Build four-option questions | free |
| 6 | Quality gate | free |

Everything writes to Google Drive, so a disconnect costs minutes not hours.
Long loops are resumable: re-run the cell and it skips what is done.

## Setup

Run first, every session. Environment variables and pip installs do not
survive a restart.

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/dataset', exist_ok=True)
os.chdir('/content/drive/MyDrive/dataset')

# always re-clone so the latest pipeline code is used
!rm -rf /tmp/eda
!git clone -q https://github.com/ZiA-rR/eda.git /tmp/eda
!cp /tmp/eda/pipeline/*.py .

!pip install -q yfinance trafilatura google-genai openai scikit-learn

# --- keys ---
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")     # serper.dev, for news retrieval
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")     # aistudio.google.com/apikey, for extraction
os.environ["GROQ_API_KEY"]     = userdata.get("GROQ_API_KEY")

import retrieval, llm
print("working in:", os.getcwd())
print("retrieval:", retrieval.VERSION)
print("llm:      ", llm.VERSION)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 16.1 MB/s eta 0:00:00
working in: /content/drive/MyDrive/dataset
retrieval: 2026-08-04-a
llm:       2026-08-04-h


Model names and free-tier quotas change often, so ask rather than assume.
`autoconfigure` lists what each provider actually serves, picks a
general-purpose chat model, and verifies it responds.

In [2]:
import os
for f in ["extracted_progress.json", "scored_progress.json"]:
    if os.path.exists(f): os.remove(f)

In [ ]:
import extraction
print("plummeted filter:", "plummeted" in open("extraction.py").read())
print("date leak check:", "_date_leaks" in open("assembly.py").read())

plummeted filter: True
date leak check: True


In [3]:
working = llm.check_models()
print("usable:", working)

  gemini-flash-lite-latest rejects thinking_budget, retrying without it
  gemini     ok   (gemini-flash-lite-latest)
  groq       ok   (llama-3.3-70b-versatile)
usable: ['gemini', 'groq']


In [4]:
working = llm.autoconfigure()
print("\nusable models:", working)

# Scoring needs at least two DIFFERENT families. Two models that share
# training agree for the wrong reasons, which tells you nothing.
if len(working) < 2:
    print("\nonly one model available. Scoring will run but there will be no "
          "disagreement signal and nothing will route to human review.")

groq       -> llama-3.3-70b-versatile
  gemini     ok   (gemini-flash-lite-latest)
  groq       ok   (llama-3.3-70b-versatile)

usable models: ['gemini', 'groq']


## Stage 1: big move days

A big move day is one where the price moved far more than is normal **for
that asset**. Crypto moving 5% is an ordinary day; a major currency pair
moving 1% is not. So the test is volatility-relative, never a fixed
percentage.

Uses the Lee-Mykland jump test (2008, Review of Financial Studies), which
estimates volatility with bipower variation so a jump cannot inflate the
very threshold it is being measured against, and takes its cutoff from a
Gumbel distribution rather than a number chosen by hand.

In [ ]:
from retrieval import TICKERS, fetch_prices

prices = {}
for asset, ticker in TICKERS.items():
    prices[asset] = fetch_prices(ticker, "2019-01-01", "2024-12-31")
    print(f"{asset:8s} {ticker:10s} {len(prices[asset])} days")

crypto   BTC-USD    2191 days
gold     GC=F       1509 days
petrol   BZ=F       1510 days
forex    GBPUSD=X   1564 days


In [ ]:
from big_moves import select_big_move_days

results = {}
for asset, p in prices.items():
    days = select_big_move_days(p, method="lee_mykland", alpha=0.01)
    results[asset] = days
    print(f"\n=== {asset.upper()} — {len(days)} big move days ===")
    print(days[["pct_move", "direction", "score"]].head(8).round(2))


=== CRYPTO — 26 big move days ===
            pct_move direction  score
Date                                 
2019-04-02     17.36        up  24.88
2020-03-12    -37.17      down  22.83
2019-09-24    -11.40      down  13.56
2023-08-17     -7.10      down  12.58
2020-07-27     10.96        up  12.20
2019-10-25     15.58        up  10.29
2019-02-18      6.58        up   9.14
2019-02-08      7.86        up   9.10

=== GOLD — 9 big move days ===
            pct_move direction  score
Date                                 
2021-06-17     -4.61      down   8.62
2021-01-08     -4.09      down   7.45
2020-11-09     -4.98      down   6.99
2019-02-19      1.67        up   6.65
2019-06-20      3.59        up   6.54
2021-01-04      2.73        up   6.23
2023-02-03     -2.79      down   6.04
2021-07-29      1.76        up   5.87

=== PETROL — 12 big move days ===
            pct_move direction  score
Date                                 
2020-03-09    -24.10      down  11.95
2019-09-16     14.61    

**Sanity check.** The flagged dates should include events you recognise:
March 2020 (covid), Feb-Mar 2022 (Ukraine), Sept 2022 (mini-budget),
Nov 2022 (FTX). If they look random, stop here.

In [ ]:
import pandas as pd

N_PER_ASSET = 15    # raise this for a bigger dataset

selected = []
for asset, days in results.items():
    for date, row in days.head(N_PER_ASSET).iterrows():
        selected.append({"asset": asset,
                         "date": date.strftime("%Y-%m-%d"),
                         "pct_move": round(row["pct_move"], 2)})

selected = pd.DataFrame(selected)
selected.to_csv("big_move_days.csv", index=False)
print(f"{len(selected)} events saved")
selected.head(10)

41 events saved


,asset,date,pct_move
0,crypto,2019-04-02,17.36
1,crypto,2020-03-12,-37.17
2,crypto,2019-09-24,-11.40
3,crypto,2023-08-17,-7.10
4,crypto,2020-07-27,10.96
5,crypto,2019-10-25,15.58
6,crypto,2019-02-18,6.58
7,crypto,2019-02-08,7.86
8,crypto,2023-10-23,10.31
9,crypto,2022-11-09,-14.35


## Stage 2: news retrieval

Articles from a window around each event. The window leans backwards
because causes precede the move.

Distractor documents are pulled with deliberately off-chain queries.
Without them the retrieval half of the task is trivial, which is why AER
did the same. Results are filtered three ways: by outlet (a whitelist of
financial press), by title (daily price tables like "Gold Rate Today"
carry no explanation), and by date.

Check one event before spending credits on the batch.

In [ ]:
from retrieval import debug_search
debug_search("gold", "2023-03-13")

backend : serper
query   : 'gold price'
date    : 2023-03-13

raw results: 8
  [keep] mining.com               Mar 13, 2023   -> 2023-03-13   Gold price surges as SVB fiasco propel
  [keep] reuters.com              Mar 13, 2023   -> 2023-03-13   Gold, silver soar as SVB collapse spur
  [keep] miningweekly.com         Mar 13, 2023   -> 2023-03-13   Pan African maximises rand gold price 
  [drop] e.vnexpress.net          Mar 12, 2023   -> 2023-03-12   Gold prices return to near historic pe
  [keep] tradingview.com          Mar 11, 2023   -> 2023-03-11   Will the price of gold continue to ris
  [drop] goldbroker.com           Mar 9, 2023    -> 2023-03-09   Gold Remains The Best Portfolio Safety
  [drop] shafaq.com               Mar 14, 2023   -> 2023-03-14   Gold prices edged higher in the Iraqi 
  [drop] thedailystar.net         Mar 9, 2023    -> 2023-03-09   Drop in gold price fails to improve sa

price-listing pages dropped: 0
after all filters: 4 of 8

testing text extraction on the f

In [ ]:
import json
from checkpoint import resumable_map, checkpoint_status
from retrieval import build_topic
import pandas as pd

selected = pd.read_csv("big_move_days.csv").to_dict("records")

def fetch(ev):
    t = build_topic(ev["asset"], ev["date"], n_relevant=15, n_distractor=5,
                    verbose=False)
    t["asset"] = ev["asset"]
    t["event_date"] = ev["date"]
    t["target_event"] = f"{ev['asset']} moved {ev['pct_move']}% on {ev['date']}"
    return t

topics = resumable_map(
    items   = selected,
    key_fn  = lambda ev: f"{ev['asset']}_{ev['date']}",
    work_fn = fetch,
    path    = "topics_progress.json",
)

for i, t in enumerate(topics):
    t["topic_id"] = i

json.dump(topics, open("topics.json", "w"), indent=2)
print(f"\n{len(topics)} topics saved")

41 to process, 0 skipped

[1/41] crypto_2019-04-02
  dropped 1 daily price-listing pages
  no whitelisted outlets among 1 results, keeping all. saw: cpajournal.com
  dropped 1 daily price-listing pages
  no whitelisted outlets among 1 results, keeping all. saw: blockchain-council.org
[2/41] crypto_2020-03-12
  no whitelisted outlets among 7 results, keeping all. saw: bitcoinke.io, businessday.ng, cpajournal.com, jdsupra.com, riskandinsurance.com, timesofmalta.com
  no whitelisted outlets among 1 results, keeping all. saw: textilegence.com
[3/41] crypto_2019-09-24
  dropped 2 daily price-listing pages
  dropped 1 daily price-listing pages
  no whitelisted outlets among 1 results, keeping all. saw: news.bitcoin.com
[4/41] crypto_2023-08-17
  dropped 1 daily price-listing pages
  no whitelisted outlets among 10 results, keeping all. saw: binance.com, crypto.com, cryptobriefing.com, guardian.ng, mb.com.ph, mckinsey.com
[5/41] crypto_2020-07-27
  dropped 1 daily price-listing pages
  droppe

In [ ]:
import statistics as st

docs = [len(t["docs"]) for t in topics]
rel  = [t.get("n_relevant", 0) for t in topics]

print(f"docs/topic: mean {st.mean(docs):.1f}, min {min(docs)}, max {max(docs)}")
print(f"on-story:   mean {st.mean(rel):.1f}")
print(f"too thin:   {sum(1 for t in topics if t.get('thin'))} of {len(topics)}")
print("\nAER reference: 19.7 docs/topic")

import shutil
shutil.copy("topics.json", "topics_backup.json")

docs/topic: mean 14.4, min 7, max 18
on-story:   mean 11.9
too thin:   0 of 41

AER reference: 19.7 docs/topic


'topics_backup.json'

## Stage 3: extract candidate causes

A model reads the articles and lists events that could explain the move.

The prompt is deliberately strict about what counts. Price readings,
percentage changes, volume statistics, technical analysis, other assets
moving in parallel, and analyst forecasts all describe the **effect**, not
a cause, and an earlier version of this returned almost nothing else.

One request per topic rather than one per document: at ~14 documents each
that is the difference between 41 requests and 574, which matters on a
free tier.

Test on one topic first. Pick one with a well-documented cause.

In [5]:
import json, statistics as st
import extraction
from checkpoint import resumable_map

topics = json.load(open("topics.json"))

# a topic with plenty of on-story documents
i = max(range(len(topics)), key=lambda k: topics[k].get("n_relevant", 0))
print(f"topic {i}: {topics[i]['asset']} {topics[i]['event_date']}\n")

t = extraction.extract_candidates_for_topic_batched(topics[i], model="gemini")
print(t["target_event"], "\n")
for cand in t["candidates"]:
    print(f"[{cand['position']:7s}] {cand['text'][:88]}")

topic 9: crypto 2022-11-09

    1 request over 18 documents -> 10 events
  10 raw -> 10 filtered -> 8 unique  (8 before, 0 after)
crypto moved -14.35% on 2022-11-09 

[before ] Binance backed out of its emergency deal to acquire rival FTX on November 9, 2022.
[before ] Most of FTX's legal and compliance staff quit on Tuesday evening, November 8, 2022.
[before ] The Securities and Exchange Commission (SEC) expanded its investigation into FTX's U.S. 
[before ] The U.S. Department of Justice seized $3.4 billion in stolen bitcoin from a Georgia real
[before ] Binance CEO Changpeng Zhao tweeted over the weekend that Binance would sell its holdings
[before ] FTX halted withdrawals from its platform on Tuesday, November 9, 2022.
[before ] SEC Chair Gensler criticized the non-compliant crypto industry amid the FTX turmoil.
[before ] Binance signed a nonbinding agreement on Tuesday, November 8, 2022, to buy FTX's non-US 


These should be discrete events with names, figures and dates. If you see
price descriptions or predictions, the filter needs work before the batch.

In [6]:
import json, statistics as st
from checkpoint import resumable_map
import extraction

extracted = resumable_map(
    items   = topics,
    key_fn  = lambda t: f"{t['asset']}_{t['event_date']}",
    work_fn = lambda t: extraction.extract_candidates_for_topic_batched(t, model="gemini"),
    path    = "extracted_progress.json",
)
json.dump(extracted, open("extracted.json", "w"), indent=2)

n = [len(t["candidates"]) for t in extracted]
print(f"\ncandidates: mean {st.mean(n):.1f}, min {min(n)}, max {max(n)}")
print(f"topics with fewer than 4: {sum(1 for x in n if x < 4)}")

41 to process, 0 skipped

[1/41] crypto_2019-04-02
    1 request over 13 documents -> 0 events
  0 raw -> 0 filtered -> 0 unique  (0 before, 0 after)
[2/41] crypto_2020-03-12
    1 request over 12 documents -> 4 events
  4 raw -> 4 filtered -> 4 unique  (4 before, 0 after)
[3/41] crypto_2019-09-24
    1 request over 8 documents -> 2 events
  2 raw -> 1 filtered -> 1 unique  (1 before, 0 after)
[4/41] crypto_2023-08-17
    1 request over 16 documents -> 4 events
  4 raw -> 4 filtered -> 3 unique  (3 before, 0 after)
[5/41] crypto_2020-07-27
    1 request over 9 documents -> 0 events
  0 raw -> 0 filtered -> 0 unique  (0 before, 0 after)
[6/41] crypto_2019-10-25
    1 request over 7 documents -> 1 events
  1 raw -> 1 filtered -> 1 unique  (1 before, 0 after)
[7/41] crypto_2019-02-18
    1 request over 10 documents -> 5 events
  5 raw -> 5 filtered -> 5 unique  (5 before, 0 after)
[8/41] crypto_2019-02-08
    1 request over 11 documents -> 8 events
  8 raw -> 8 filtered -> 6 unique  (6 be

## Stage 4: score the candidates

Each candidate rated 0 to 3 for how strongly it caused the target event.
Anything scoring 2 or more becomes a correct option.

Scored by every available model. Where they agree the label is settled;
where they disagree by 2 or more it routes to human review. That queue is
the point of using several models, and it is what a Krippendorff alpha
gets computed on later.

Four coarse levels rather than a 0-100 scale is deliberate: CRAB used
0-100 and their annotators only reached 0.28 agreement, while AER used
three levels and reached 0.51.

In [7]:
import json, scoring
from checkpoint import resumable_map

extracted = json.load(open("extracted.json"))

scored = resumable_map(
    items   = extracted,
    key_fn  = lambda t: f"{t['asset']}_{t['event_date']}",
    work_fn = lambda t: scoring.score_topic_batched_multi(t, models=working),
    path    = "scored_progress.json",
)
json.dump(scored, open("scored.json", "w"), indent=2, default=str)

for k, v in scoring.score_report(scored).items():
    print(f"  {k:32s} {v}")

41 to process, 0 skipped

[1/41] crypto_2019-04-02
[2/41] crypto_2020-03-12
  2 models x 1 request. scores {0: 2, 1: 1, 2: 1}, 0 need review
[3/41] crypto_2019-09-24
  2 models x 1 request. scores {0: 1}, 0 need review
[4/41] crypto_2023-08-17
  2 models x 1 request. scores {1: 1, 2: 2}, 1 need review
[5/41] crypto_2020-07-27
[6/41] crypto_2019-10-25
  2 models x 1 request. scores {2: 1}, 1 need review
[7/41] crypto_2019-02-18
  2 models x 1 request. scores {0: 4, 3: 1}, 0 need review
[8/41] crypto_2019-02-08
  2 models x 1 request. scores {0: 2, 1: 2, 2: 2}, 2 need review
[9/41] crypto_2023-10-23
  2 models x 1 request. scores {2: 1}, 1 need review
[10/41] crypto_2022-11-09
  2 models x 1 request. scores {0: 3, 1: 1, 2: 2}, 1 need review
[11/41] crypto_2019-10-23
[12/41] crypto_2022-06-13
  2 models x 1 request. scores {3: 1}, 0 need review
[13/41] crypto_2022-01-21
  2 models x 1 request. scores {0: 1, 1: 1, 2: 1, 3: 1}, 0 need review
[14/41] crypto_2023-01-12
  2 models x 1 request.

In [8]:
import json, scoring
from checkpoint import resumable_map

extracted = json.load(open("extracted.json"))

scored = resumable_map(
    items   = extracted,
    key_fn  = lambda t: f"{t['asset']}_{t['event_date']}",
    work_fn = lambda t: scoring.score_topic_batched_multi(t, models=working),
    path    = "scored_progress.json",
)
json.dump(scored, open("scored.json", "w"), indent=2, default=str)

for k, v in scoring.score_report(scored).items():
    print(f"  {k:32s} {v}")

resuming from scored_progress.json: 41 done, 0 failed
0 to process, 41 skipped


41 done, 0 failed, saved to scored_progress.json
  topics                           41
  candidates                       149
  score_distribution               {0: 46, 1: 31, 2: 41, 3: 31}
  unscored                         0
  needing_review                   73
  topics_with_no_correct_answer    13
  topics_with_under_4_candidates   22
  usable_topics                    21


Transient failures leave candidates unscored, and an unscored candidate
cannot be a correct answer. This retries only the affected topics.

In [9]:
import json

scored = scoring.rescore_unscored(scored, models=working)
json.dump(scored, open("scored.json", "w"), indent=2, default=str)

print("\nafter retry:")
for k, v in scoring.score_report(scored).items():
    print(f"  {k:32s} {v}")

0 topics have 0 unscored candidates


after retry:
  topics                           41
  candidates                       149
  score_distribution               {0: 46, 1: 31, 2: 41, 3: 31}
  unscored                         0
  needing_review                   73
  topics_with_no_correct_answer    13
  topics_with_under_4_candidates   22
  usable_topics                    21


## Stage 5: build the questions

Correct options are anything scoring 2 or 3. Distractors are stratified
into AER's three types (temporal, semantic, background) and chosen so
their lengths sit close to the correct options.

Where a topic does not yield enough of its own distractors, it borrows
candidates from **other events of the same asset**. A real cause of a
different gold move is topically convincing but definitively not a cause
of this one. Borrowed options are restricted to events within a few months
and screened for dates that would give them away.

In [10]:
import json, random, assembly

# if resuming here rather than running straight through
try:
    scored
except NameError:
    scored = json.load(open("scored.json"))

rng = random.Random(42)
pool = assembly.build_distractor_pool(scored)
print("distractor pool per asset:", {k: len(v) for k, v in pool.items()})

questions = []
for t in scored:
    q = assembly.build_question(t, pool=pool, rng=rng)
    if q:
        q["id"] = f"{t['asset']}_{t['event_date']}"
        questions.append(q)
questions = assembly.rebalance_positions(questions, rng)

print(f"\n{len(questions)} questions from {len(scored)} topics")
print("label consistency:", assembly.verify_consistency(questions)["n_bad"], "problems")

distractor pool per asset: {'crypto': 44, 'gold': 15, 'petrol': 69, 'forex': 21}

16 questions from 41 topics
label consistency: 0 problems


## Stage 6: quality gate

Two kinds of shortcut, both found in AER.

**Style leakage.** A classifier reading only the option text, with no
question and no documents, scored 89.4% on AER against a 60.6% baseline.
Most of the answer sat in how options were phrased.

**Structural leakage.** The winning AER system gained 5.6 points from
rules applied after the model answered, because the "none of the others"
option was correct every time it appeared and duplicate options always
shared a truth value.

The leakage classifier needs a few hundred questions before its number
means anything, so it reports low confidence on a small sample. The length
and position checks apply at any size.

In [11]:
import json, quality

try:
    questions
except NameError:
    questions = [json.loads(l) for l in open("sample_questions.jsonl")]

report = quality.run_gate(questions)

QUALITY GATE  (16 questions)
  style leakage    61.5% vs 53.8% baseline (+7.7pp)
                   only 16 questions, needs ~200+ before this number means much
  option lengths   correct 23.8 vs wrong 21 words (gap 2.82)
  word overlap     correct 0.0624 vs wrong 0.0511
  'none' option    present 12.5%, correct 0.0% of those
  duplicates       0.0%
  positions        {'A': 9, 'B': 8, 'C': 8, 'D': 9} (spread 2.9%)
  cardinality      {1: 37.5, 2: 12.5, 3: 50.0}
--------------------------------------------------------------
  RESULT: PASSED


In [ ]:
import json

with open("sample_questions.jsonl", "w") as f:
    for q in questions:
        f.write(json.dumps(q) + "\n")
print(f"saved {len(questions)} questions")

saved 18 questions


**Read the questions.** The gate checks statistics, not sense. It cannot
tell you whether a marked answer is really a cause, or whether a borrowed
distractor is obviously wrong.

In [12]:
for q in questions[:3]:
    print("=" * 70)
    print(q["id"], "|", q["target_event"], "\n")
    for L in "ABCD":
        mark = "*" if L in q["golden_answer"] else " "
        print(f"{mark} {L}. [{q['causal_strength'][L]}] ({q['option_types'][L]})")
        print(f"     {q[f'option_{L}']}")
    print(f"\n  answer: {q['golden_answer']}\n")

crypto_2019-10-25 | crypto moved 15.58% on 2019-10-25 

  A. [0] (semantic_crosstopic)
     Minutes from the Federal Reserve’s December Federal Open Market Committee showed officials were planning to hike interest rates sooner than expected during an inflation surge.
  B. [0] (semantic_crosstopic)
     Binance CEO Changpeng Zhao announced on Twitter over the weekend that Binance would sell its holdings of FTT.
* C. [2] (correct)
     On Wednesday, Facebook CEO Mark Zuckerberg testified before the House Financial Services Committee regarding the company's cryptocurrency plans for Libra on 2019-10-23.
  D. [0] (none_option)
     None of the other options are correct causes.

  answer: C

crypto_2019-02-08 | crypto moved 7.86% on 2019-02-08 

  A. [1] (background)
     On February 7, 2019, the Litecoin Foundation and Beam Corporation announced a possible collaboration that would use Beam's Mimblewimble protocol to improve security.
* B. [2] (correct)
     Securities and Exchange Commissio

## Human review queue

Candidates the models disagreed on. Everyone labels a shared subset, which
is what Krippendorff's alpha is computed on, and the rest is split. With
three annotators and 15% overlap each person sees about 43% of the queue.

Alpha needs at least two annotators to exist at all. For reference: AER
reported 0.51, CRAB's expert reviewers 0.70, UNcommonsense 0.40 to 0.60.

In [13]:
import json, finalise, scoring

queue = scoring.review_queue(scored)
print(f"{len(queue)} candidates need review\n")
for item in queue[:5]:
    print(f"  spread {item['spread']}  {item['model_scores']}")
    print(f"    {item['candidate'][:88]}\n")

if queue:
    info = finalise.make_review_sheet(queue, "review_sheet.json",
                                      overlap_frac=0.15, n_annotators=3)
    print("shared subset:", info["overlap_items"])
    print("per annotator:", info["per_annotator"])

73 candidates need review

  spread 2  {'gemini': 2, 'groq': 0}
    U.S. interest rates hit multi-year highs on 17 August, with the 30-year Treasury bond ri

  spread 2  {'gemini': 1, 'groq': 3}
    On Wednesday, Facebook CEO Mark Zuckerberg testified before the House Financial Services

  spread 2  {'gemini': 2, 'groq': 0}
    The Securities and Exchange Commission reportedly expanded its investigation into FTX's 

  spread 2  {'gemini': 0, 'groq': 2}
    Russia's Ministry of Finance proposed legislation on 2020-09-03 to restrict the circulat

  spread 2  {'gemini': 1, 'groq': 3}
    Col. Turki al-Maliki, spokesperson for the Saudi-led Coalition to Restore Legitimacy in 

shared subset: 10
per annotator: {'annotator_1': 31, 'annotator_2': 31, 'annotator_3': 31}


## Splits and EDA

Split by **event**, not by question, so no market event appears in two
splits. Two questions about the same event share documents, so splitting
by question would leak evidence between train and test.

The cardinality mix is kept similar across splits. AER's test set was much
easier than its dev set (18.3% multi-answer against 47.5%), which makes
the two hard to compare.

In [14]:
import json, finalise

splits = finalise.make_splits(questions)

for name, qs in splits.items():
    with open(f"{name}.jsonl", "w") as f:
        for q in qs:
            f.write(json.dumps(q) + "\n")

finalise.print_eda(finalise.dataset_eda(questions, topics))

split     events  questions  multi-answer
------------------------------------------
train          9          9         66.7%
dev            1          1        100.0%
test           6          6         50.0%

(AER's test split was much easier than its dev split, 18.3% vs 47.5% multi-answer. These should be close.)
DATASET EDA
  questions          16
  events             16 (1.0 questions each)
  by asset           {'crypto': 5, 'gold': 2, 'petrol': 8, 'forex': 1}
  cardinality        {1: 37.5, 2: 12.5, 3: 50.0}
  multi-answer       62.5%   (AER 43.58%)
  mean gold set      2.125   (AER 1.574)

  docs per topic     14.39   (AER 19.7)
  tokens per topic   ~9350   (AER 28047)
  unique sources     150   (AER 535)
  top 5 source share 29.0%   (AER 21%)
  distractor docs    17.5%


## Evaluating models on it

This is what the dataset is for. The headline score matters less than the
breakdown: the AILS-NTUA team found under-selections outnumbered
over-selections 1,389 to 52 across 14 models from 7 families, and accuracy
collapsed as the number of correct answers rose.

Whether the same pattern shows up in financial reasoning is the finding
worth reporting.

In [15]:
import json, evaluate

docs_by_topic = {t["topic_id"]: t for t in topics if "topic_id" in t}

evals = []
for m in working:
    print(f"\nevaluating {m}")
    evals.append(evaluate.evaluate_model(splits["test"], docs_by_topic, model=m))

if evals:
    print("\n" + evaluate.compare_models(evals))

for ev in evals:
    fa = evaluate.failure_analysis(ev, splits["test"])
    uo = fa["under_vs_over"]
    print(f"\n=== {ev['model']} ===")
    print(f"  under-selected {uo['under_selected']}, over-selected {uo['over_selected']}")
    print(f"  ({uo['aer_reference']})")
    for k, v in fa["by_cardinality"].items():
        print(f"    {k} correct: exact {v['exact_match']:.2f}, F1 {v['f1']:.2f}, "
              f"predicted {v['mean_predicted']} on average")


evaluating gemini

evaluating groq

model            n    exact       F1     prec   recall    pred/gold
------------------------------------------------------------------
gemini           6    0.667    0.861    0.833    0.944  2.33/2.00  
groq             6    0.000    0.000    0.000    0.000  0.00/2.00  

=== gemini ===
  under-selected 1, over-selected 3
  (1389 vs 52 in the AER winning system's analysis)
    1 correct: exact 0.67, F1 0.83, predicted 1.67 on average
    3 correct: exact 0.67, F1 0.89, predicted 3 on average

=== groq ===
  under-selected 12, over-selected 0
  (1389 vs 52 in the AER winning system's analysis)
    1 correct: exact 0.00, F1 0.00, predicted 0 on average
    3 correct: exact 0.00, F1 0.00, predicted 0 on average


## Troubleshooting

Re-run any long cell after a disconnect: the checkpoint skips what is done.

- **`limit: 0` or a daily quota error** — that model has no allowance left.
  Run `llm.autoconfigure()` to find one that does.
- **429 from Groq** — bursts trip it. Raise `GROQ_INTERVAL`.
- **Empty Gemini response** — thinking tokens ate the budget. Already
  disabled for 2.5+, but raise `max_tokens` if it recurs.
- **A model name 404s** — catalogues churn. `llm.list_provider_models()`
  shows what a provider actually serves today.

In [16]:
from checkpoint import checkpoint_status, show_failures

for f in ["topics_progress.json", "extracted_progress.json", "scored_progress.json"]:
    try:
        print(checkpoint_status(f))
    except Exception:
        print(f"{f}: not started")

{'path': 'topics_progress.json', 'exists': True, 'done': 41, 'failed': 0, 'failed_keys': []}
{'path': 'extracted_progress.json', 'exists': True, 'done': 41, 'failed': 0, 'failed_keys': []}
{'path': 'scored_progress.json', 'exists': True, 'done': 41, 'failed': 0, 'failed_keys': []}


In [ ]:
# what each provider serves today
llm.list_provider_models("groq")
# llm.list_gemini_models()

In [ ]:
from checkpoint import checkpoint_status
print(checkpoint_status("scored_progress.json"))

import llm
working = llm.check_models()
print("usable:", working)

{'path': 'scored_progress.json', 'exists': True, 'done': 8, 'failed': 0, 'failed_keys': []}
  gemini     ok   (gemini-flash-lite-latest)
  groq       ok   (llama-3.3-70b-versatile)
usable: ['gemini', 'groq']
